<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/week7/Semana_7_d%C3%ADa_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predecir precios de productos

### Semana 7 Día 1

Introducción a LoRA y QLoRA

In [1]:
# pip installs
# peft - ajuste fino de parámetros

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers==4.43.1 trl accelerate sentencepiece

In [1]:
# imports

import os
# Proporciona funciones para interactuar con el sistema operativo (ej. acceder a variables de entorno)

import re
# Módulo de expresiones regulares para búsqueda y manipulación de patrones en texto

import math
# Funciones matemáticas básicas (sin, cos, sqrt, etc.)

from tqdm import tqdm
# Barra de progreso visual para loops — muestra avance y tiempo estimado durante iteraciones

from google.colab import userdata
# Acceso a secretos guardados en Google Colab (ej. tokens de autenticación) de forma segura

from huggingface_hub import login
# Autenticarse en Hugging Face Hub usando token de API — permite descargar modelos privados y subir resultados

import torch
# Framework de deep learning de PyTorch — gestión de tensores, GPU computing, autograd para gradientes

import transformers
# Librería de Hugging Face con modelos pre-entrenados (BERT, GPT, Llama, etc.) y utilidades

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
# Funciones específicas de transformers:
#   - AutoModelForCausalLM: carga automáticamente modelos generativos (ej. Llama 3.1) según el nombre
#   - AutoTokenizer: carga automáticamente el tokenizador correspondiente al modelo
#   - BitsAndBytesConfig: configuración para cuantización de 8 bits y 4 bits (NF4) durante carga del modelo
#   - TrainingArguments: especifica parámetros de entrenamiento (learning rate, batch size, épocas, etc.)
#   - set_seed: fija la semilla aleatoria para reproducibilidad

from peft import LoraConfig, PeftModel
# PEFT = Parameter-Efficient Fine-Tuning (Hugging Face):
#   - LoraConfig: define la configuración de LoRA (r, alpha, target_modules)
#   - PeftModel: wrapper que aplica los adaptadores LoRA a un modelo base congelado

from datetime import datetime
# Manejo de fechas y tiempos — para registrar timestamps (cuándo empezó el entrenamiento, etc.)

In [2]:
# Constants

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = f"ed-donner/pricer-2024-09-13_13.04.39"

# Hyperparameters para Fine-Tuning QLoRA

LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

### Inicia sesión en HuggingFace

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

In [3]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Probando diferentes Cuantizaciones


In [5]:
# Cargar el modelo base sin cuantizar

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [6]:
print(f"Impacto en la memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

Impacto en la memoria: 32.1 GB


In [5]:
base_model

NameError: name 'base_model' is not defined

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [4]:
# Cargar el modelo base en 8 bits

quant_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
print(f"Impacto en Memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

Impacto en Memoria: 9.1 GB


In [9]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear8bitLt(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [10]:
# !pip install --upgrade transformers bitsandbytes accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 80.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.1
    Uninstalling hf-xet-1.5.1:
      Successfully uninstalled hf-xet-1.5.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.43.1
 

In [9]:
# Cargar el Tokenizer y el modelo Base en 4 bit

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    #device_map="auto",
)

`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [ ]:
print(f"Impacto en memoria: {base_model.get_memory_footprint() / 1e9:,.2f} GB")

Impacto en memoria: 5.59 GB


In [ ]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)

adapter_config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/109M [00:00<?, ?B/s]

In [ ]:
print(f"Impacto en memoria: {fine_tuned_model.get_memory_footprint() / 1e9:,.2f} GB")

Impacto en memoria: 5.70 GB


In [ ]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [ ]:
# Cada uno de los módulos de destino tiene 2 matrices de adaptador LoRA, llamadas lora_A y lora_B
# Están diseñadas para que los pesos se puedan adaptar sumando alpha * lora_A * lora_B
# Contemos la cantidad de pesos usando sus dimensiones:

# Vea las dimensiones de la matriz anterior
lora_q_proj = 4096 * 32 + 4096 * 32
lora_k_proj = 4096 * 32 + 1024 * 32
lora_v_proj = 4096 * 32 + 1024 * 32
lora_o_proj = 4096 * 32 + 4096 * 32

# Cada capa utiliza pues
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# Hay en total 32 capas
params = lora_layer * 32

# Por tanto el total en Mb es:
size = (params * 4) / 1_000_000

print(f"Número total de parámetros: {params:,} y tamaño {size:,.1f}MB")

Número total de parámetros: 27,262,976 y tamaño 109.1MB
